# train_repo_models_mlflow


In [0]:
%pip install -q transformers torch accelerate mlflow scikit-learn pandas cloudpickle
dbutils.library.restartPython()

In [0]:
import os
import shutil
import tempfile
import pandas as pd
import numpy as np
import torch
import mlflow
import mlflow.pyfunc

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from mlflow.models import infer_signature

### Configurar experimento y Tabla

In [0]:
catalog = "workspace"
schema = "financial_sentiment"
table_prefix = f"{catalog}.{schema}"

experiment_name = "/Users/jezapataf@eafit.edu.co/financial_sentiment_mlops"
mlflow.set_experiment(experiment_name)
mlflow.set_registry_uri("databricks-uc")

df = spark.table(f"{table_prefix}.gold_financial_sentiment_training").toPandas()

print("Total registros:", len(df))
print(df["split"].value_counts())
print(df["label_normalized"].value_counts())

### Definir Modelos

In [0]:
MODEL_SPECS = [
    {
        "model_key": "finbert",
        "model_name": "ProsusAI/finbert",
        "label_order": ["positive", "negative", "neutral"]
    },
    {
        "model_key": "distilroberta_financial",
        "model_name": "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis",
        "label_order": ["negative", "neutral", "positive"]
    }
]

In [0]:
test_df = df[df["split"] == "test"][["text_clean", "label_normalized"]].dropna().copy()

SAMPLE_PER_CLASS = 300

eval_df = (
    test_df
    .groupby("label_normalized", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), SAMPLE_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)

print("Registros evaluación:", len(eval_df))
print(eval_df["label_normalized"].value_counts())
display(eval_df.head())

In [0]:
def predict_transformer(model, tokenizer, texts, label_order, max_length=128, batch_size=16):
    all_predictions = []
    all_scores = []

    model.eval()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=1)
            pred_ids = torch.argmax(probs, dim=1).cpu().numpy()
            scores = torch.max(probs, dim=1).values.cpu().numpy()

        batch_predictions = [label_order[idx] for idx in pred_ids]

        all_predictions.extend(batch_predictions)
        all_scores.extend(scores.tolist())

    return pd.DataFrame({
        "prediction": all_predictions,
        "score": all_scores
    })

In [0]:
class HuggingFaceSentimentPyFunc(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        self.model_dir = context.artifacts["hf_model"]
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()

        metadata_path = os.path.join(self.model_dir, "mlflow_label_order.txt")
        with open(metadata_path, "r") as f:
            self.label_order = [line.strip() for line in f.readlines() if line.strip()]

    def predict(self, context, model_input):
        import pandas as pd
        import torch

        if isinstance(model_input, pd.DataFrame):
            texts = model_input["text_clean"].astype(str).tolist()
        else:
            texts = pd.DataFrame(model_input)["text_clean"].astype(str).tolist()

        all_predictions = []
        all_scores = []

        batch_size = 16
        max_length = 128

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            with torch.no_grad():
                outputs = self.model(**encoded)
                probs = torch.softmax(outputs.logits, dim=1)
                pred_ids = torch.argmax(probs, dim=1).cpu().numpy()
                scores = torch.max(probs, dim=1).values.cpu().numpy()

            batch_predictions = [self.label_order[idx] for idx in pred_ids]

            all_predictions.extend(batch_predictions)
            all_scores.extend(scores.tolist())

        return pd.DataFrame({
            "prediction": all_predictions,
            "score": all_scores
        })

In [0]:
class HuggingFaceSentimentPyFunc(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        self.model_dir = context.artifacts["hf_model"]
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()

        metadata_path = os.path.join(self.model_dir, "mlflow_label_order.txt")
        with open(metadata_path, "r") as f:
            self.label_order = [line.strip() for line in f.readlines() if line.strip()]

    def predict(self, context, model_input):
        import pandas as pd
        import torch

        if isinstance(model_input, pd.DataFrame):
            texts = model_input["text_clean"].astype(str).tolist()
        else:
            texts = pd.DataFrame(model_input)["text_clean"].astype(str).tolist()

        all_predictions = []
        all_scores = []

        batch_size = 16
        max_length = 128

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            with torch.no_grad():
                outputs = self.model(**encoded)
                probs = torch.softmax(outputs.logits, dim=1)
                pred_ids = torch.argmax(probs, dim=1).cpu().numpy()
                scores = torch.max(probs, dim=1).values.cpu().numpy()

            batch_predictions = [self.label_order[idx] for idx in pred_ids]

            all_predictions.extend(batch_predictions)
            all_scores.extend(scores.tolist())

        return pd.DataFrame({
            "prediction": all_predictions,
            "score": all_scores
        })

In [0]:
for spec in MODEL_SPECS:
    model_key = spec["model_key"]
    model_name = spec["model_name"]
    label_order = spec["label_order"]

    print(f"Procesando modelo: {model_key} - {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    X_eval = eval_df[["text_clean"]].copy()
    y_true = eval_df["label_normalized"].tolist()

    pred_df = predict_transformer(
        model=model,
        tokenizer=tokenizer,
        texts=X_eval["text_clean"].astype(str).tolist(),
        label_order=label_order,
        max_length=128,
        batch_size=16
    )

    y_pred = pred_df["prediction"].tolist()

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)

    report_dict = classification_report(y_true, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()

    confusion_df = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=["negative", "neutral", "positive"]),
        index=["real_negative", "real_neutral", "real_positive"],
        columns=["pred_negative", "pred_neutral", "pred_positive"]
    )

    with tempfile.TemporaryDirectory() as tmpdir:
        local_model_dir = os.path.join(tmpdir, model_key)
        os.makedirs(local_model_dir, exist_ok=True)

        tokenizer.save_pretrained(local_model_dir)
        model.save_pretrained(local_model_dir)

        with open(os.path.join(local_model_dir, "mlflow_label_order.txt"), "w") as f:
            for label in label_order:
                f.write(label + "\n")

        report_path = os.path.join(tmpdir, f"{model_key}_classification_report.csv")
        confusion_path = os.path.join(tmpdir, f"{model_key}_confusion_matrix.csv")
        predictions_path = os.path.join(tmpdir, f"{model_key}_sample_predictions.csv")

        report_df.to_csv(report_path)
        confusion_df.to_csv(confusion_path)

        predictions_output = eval_df.copy()
        predictions_output["prediction"] = y_pred
        predictions_output["score"] = pred_df["score"].values
        predictions_output.to_csv(predictions_path, index=False)

        input_example = X_eval.head(5)
        output_example = pred_df.head(5)

        signature = infer_signature(
            model_input=input_example,
            model_output=output_example
        )

        with mlflow.start_run(run_name=f"repo_model_{model_key}") as run:
            mlflow.log_param("source", "repo_original")
            mlflow.log_param("model_key", model_key)
            mlflow.log_param("hf_model_name", model_name)
            mlflow.log_param("max_length", 128)
            mlflow.log_param("eval_sample_per_class", SAMPLE_PER_CLASS)
            mlflow.log_param("label_order", ",".join(label_order))
            mlflow.log_param("input_column", "text_clean")

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("macro_f1", macro_f1)
            mlflow.log_metric("weighted_f1", weighted_f1)

            mlflow.log_artifact(report_path)
            mlflow.log_artifact(confusion_path)
            mlflow.log_artifact(predictions_path)

            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=HuggingFaceSentimentPyFunc(),
                artifacts={"hf_model": local_model_dir},
                signature=signature,
                input_example=input_example,
                pip_requirements=[
                    "mlflow",
                    "transformers",
                    "torch",
                    "pandas",
                    "cloudpickle"
                ]
            )

            print("Run ID:", run.info.run_id)
            print("Modelo logueado:", model_key)